# A clinical benchmark for compact RNA models: ClinVar variants in non-coding RNA genes

**The question.** Can a few-million-parameter model score clinically reported variants as well as a 650M RNA
language model, with no labels and no fine-tuning?

**Why this task.** It is the only clinical endpoint a *sequence* model can be put on directly. Patient-level AML
tasks read expression matrices, so a 650M language model and a 5M student cannot be compared on them without an
arbitrary bridge. Here both score the same string, by the same rule:

$$s = \log p(\text{ref}\mid\text{context, position masked}) - \log p(\text{alt}\mid\ldots)$$

The genes that survive the filters are real disease genes: **RNU4-2** (neurodevelopmental disorder), **RNU4ATAC**
(Roifman / Taybi-Linder), **RMRP** (cartilage-hair hypoplasia), **SNORD118** (leukoencephalopathy), **RNU12**,
**RNU2-2**, **RNU7-1**, and mitochondrial tRNAs (**MT-TL1** is the MELAS gene).

**What this notebook produces**
1. The labelled panel, built from ClinVar + GENCODE with the attribution and coding-consequence filters that stop
   the label belonging to a neighbouring protein-coding gene.
2. A **teacher size ladder**: RiNALMo 33M / 150M / 650M scored zero-shot on identical rows. Does scale buy clinical
   accuracy?
3. **Distilled compact students** (~1M and ~5M) trained on human non-coding transcripts with the panel's genes
   removed, in three arms that differ only in the target: the true base (`mlm`, i.e. train your own small LM),
   the teacher's distribution (`kd`), or both.
4. No-model baselines, because a language model that cannot beat "transversions are worse" is not doing biology.

Gates are in `PILOT_PROTOCOL.md` and `rnakd/clinical.py`. **Gate C1 (masked-base recovery) must pass before any
score is believed** - it is what catches a mis-wired mask head.

Runtime: **A100**. Roughly 15 min of downloads, 10 min for the teacher ladder, 10 min for the transfer cache and
1-2 h for the students. Every stage is resumable.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
from google.colab import files, drive
up = files.upload()                      # choose rnakd_pilot.zip
!unzip -q -o rnakd_pilot.zip -d /content
%cd /content/rnakd_pilot

try:
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/rna_distill/clinical'
except Exception as e:
    print('Drive not mounted (%s); results are lost on disconnect' % e)
    WORK = '/content/clinwork'
import os; os.makedirs(WORK, exist_ok=True); print('WORK =', WORK)

In [ ]:
!pip -q install 'numpy>=1.26' 'pandas>=2'
# plumbing rehearsal: synthetic panel, random tiny model, no network. ~3 min.
!python -m rnakd.clinical selftest --work /content/clin_selftest

## 1. The three pretrained checkpoints
These are the **pretrained** (masked-LM) weights, not the task fine-tunes: zero-shot variant scoring uses the
language-model head directly. Zenodo record 15043668, weights CC BY 4.0. 3.3 GB in total.

In [ ]:
import os
Z = 'https://zenodo.org/records/15043668/files'
CK = {}
for cfg, fn in [('micro','rinalmo_micro_pretrained.pt'),
                ('mega','rinalmo_mega_pretrained.pt'),
                ('giga','rinalmo_giga_pretrained.pt')]:
    CK[cfg] = f'{WORK}/{fn}'
    if not os.path.exists(CK[cfg]):
        !wget -q --show-progress -O {CK[cfg]} '{Z}/{fn}?download=1'
!ls -lh {WORK}/*.pt
# look before trusting: the pretrained files should be BARE (embedding.*, transformer.*, lm_mask_head.*)
!python -m rnakd.pilot dumpkeys --ckpt-ft {CK['micro']} 2>&1 | head -8

## 2. Build the clinical panel
Downloads GENCODE (annotation + transcript sequences) and the ClinVar GRCh38 VCF, about 250 MB, then applies the
filters. The printed per-gene table is the benchmark; the reference-base check at the end is a hard correctness
test of the genomic-to-transcript mapping.

In [ ]:
!python -m rnakd.clinical panel --work {WORK}

## 3. GATE C1 + the teacher size ladder
Each cell scores every variant zero-shot and first checks that the model can recover a masked nucleotide well
above chance (0.25). A failure here means the mask head is wired wrongly, and the cell stops rather than
producing a plausible-looking number.

In [ ]:
!python -m rnakd.clinical score --work {WORK} --ckpt-lm {CK['micro']} --cfg micro --bs 32

In [ ]:
!python -m rnakd.clinical score --work {WORK} --ckpt-lm {CK['mega']}  --cfg mega  --bs 32

In [ ]:
!python -m rnakd.clinical score --work {WORK} --ckpt-lm {CK['giga']}  --cfg giga  --bs 16

## 4. The free transfer set
Human non-coding transcripts, cut into 256-nt windows, **with every panel gene removed** - without that exclusion
the student would be trained on the sequences it is about to be tested on. The teacher is run once over them and
its 22-way distribution is cached at a fixed set of masked positions, so all three student arms see identical
inputs and identical positions and differ only in the target.

In [ ]:
!python -m rnakd.clinical transfer --work {WORK} --ckpt-lm {CK['giga']} --cfg giga --bs 48 --max-n 60000

## 5. Students
`mlm` is the honest baseline: the same small model trained on the same sequences with no teacher at all. `kd` sees
only the teacher's distribution. If `kd` does not beat `mlm`, distillation bought nothing that the small model
could not learn by itself. Resumable: re-run to continue.

In [ ]:
!python -m rnakd.clinical distil --work {WORK} --presets s1,s5 --seeds 0,1,2 --epochs 3

## 6. Report

In [ ]:
!python -m rnakd.clinical report --work {WORK}

In [ ]:
import zipfile, os
with zipfile.ZipFile('/content/clinical_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('scores.csv', 'panel_summary.csv', 'clinical_report.txt', 'panel.csv.gz'):
        p = os.path.join(WORK, f)
        if os.path.exists(p):
            z.write(p, f); print('packed', f, os.path.getsize(p), 'bytes')
files.download('/content/clinical_results.zip')

## Unattended: run what is left, then shut the session down
Use this **instead of** the stage cells above when you want to walk away. It runs the remaining stages in order,
stops at the first failure without terminating, flushes Drive, prints the report, and only then releases the
runtime so the session stops drawing compute units. Everything lives on Drive, so a termination loses nothing.

In [ ]:
#@title Run the remaining stages, flush to Drive, then release the runtime
# Terminates ONLY if every stage exited 0 and the artifacts are on Drive. A failure leaves the runtime up so you
# can look at it. You get 60 s to interrupt before the session ends.
import os, subprocess, time

STAGES = [
    ['python','-m','rnakd.clinical','transfer','--work',WORK,'--ckpt-lm',CK['giga'],'--cfg','giga','--bs','48','--max-n','60000'],
    ['python','-m','rnakd.clinical','distil','--work',WORK,'--presets','s1,s5','--seeds','0,1,2','--epochs','3'],
    ['python','-m','rnakd.clinical','report','--work',WORK],
]
NEED = ['scores.csv', 'clinical_report.txt', 'panel.csv.gz']

ok, t0 = True, time.time()
for cmd in STAGES:
    print('\n>>>', ' '.join(cmd), flush=True)
    if subprocess.run(cmd).returncode != 0:
        ok = False
        print('\nSTAGE FAILED -- leaving the runtime up so you can inspect it.')
        break

missing = [f for f in NEED if not os.path.exists(os.path.join(WORK, f))]
on_drive = WORK.startswith('/content/drive')
print(f'\nelapsed {(time.time()-t0)/60:.1f} min | stages ok: {ok} | missing: {missing or "none"} | on Drive: {on_drive}')

if ok and not missing and on_drive:
    print(open(os.path.join(WORK, 'clinical_report.txt')).read())
    from google.colab import drive, runtime
    drive.flush_and_unmount()                 # force the Drive upload before the VM goes away
    print('\nResults are on Drive at', WORK)
    print('Terminating in 60 s -- press the stop button to cancel.')
    time.sleep(60)
    runtime.unassign()
else:
    print('NOT terminating. Fix the above, or if WORK is not on Drive, download the files first.')